# 03 — Risk-Neutral NIG Calibration

This notebook applies the calibration engine in `src/calibration.py` to the cleaned SPX European call sample. It presents the objective, parameter constraints, estimated parameters and fit diagnostics. The numerical implementation remains in the Python module, not in notebook cells.


## 1. Objective function and identifiable parameters

Following the assignment, we minimise the unweighted mean-square price error:

`MSE(alpha,beta,delta) = (1/N) * sum_i (C_model_i - C_market_i)^2`.

The mean-correcting risk-neutral construction removes `mu` from the option-price characteristic function. We therefore estimate `(alpha, beta, delta)` only. The admissible region is `alpha > 0`, `delta > 0`, `abs(beta) < alpha`, and `abs(beta + 1) < alpha`.


## 2. Load the cleaned calibration sample

Load the compact, long-format file committed in `data/cleaned/`. Each observation is one call quote. The calibration engine checks all required columns and rejects missing or non-finite inputs.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from src.calibration import NIGCalibrator

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'cleaned' / 'calibration_candidates.csv.gz'
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Calibration candidates not found: {DATA_PATH}')
options = pd.read_csv(DATA_PATH)
print(f'Calibration observations: {len(options):,}')
print(f'Distinct expiries: {options["Expiry"].nunique()}')
display(options.groupby('Expiry').size().rename('Number of calls').to_frame())


## 3. Configure and run the calibration

The optimizer works in transformed coordinates so every trial parameter set respects the risk-neutral exponential-moment constraints. The vectorised Gauss-Legendre pricing routine evaluates all options together at each objective-function call. The starting values are numerical initial guesses, not assumed final estimates.


In [ ]:
calibrator = NIGCalibrator(
    integration_limit=150.0,
    quadrature_points=256,
    max_nfev=500,
)
calibration_result = calibrator.calibrate(
    options,
    initial_parameters=(8.0, -2.0, 0.25),
)
print(calibration_result)


## 4. Report estimated parameters and fit statistics

The table reports the estimated risk-neutral NIG parameters and the in-sample MSE/RMSE. A successful optimizer termination is necessary but not sufficient: the next cells inspect residuals and pricing errors by maturity before we interpret the fit.


In [ ]:
estimated = calibration_result.parameters
parameter_summary = pd.Series({
    'alpha': estimated.alpha,
    'beta': estimated.beta,
    'delta': estimated.delta,
    'mu (not identified under this convention)': estimated.mu,
    'MSE': calibration_result.mean_squared_error,
    'RMSE': calibration_result.root_mean_squared_error,
    'Number of quotes': calibration_result.number_of_quotes,
    'Function evaluations': calibration_result.function_evaluations,
    'Optimizer success': calibration_result.success,
}, name='Estimate')
display(parameter_summary.to_frame())
print(calibration_result.message)


## 5. Inspect model prices and residuals

Reprice the full sample using the estimated parameters. Residuals are defined as model price minus market mid-price. Inspect both the aggregate error and the maturity-level distribution; large residuals may reveal model misspecification, noisy quotes or numerical integration issues.


In [ ]:
model_prices = calibrator.model_prices(
    options,
    estimated.alpha,
    estimated.beta,
    estimated.delta,
)
fit = options.copy()
fit['Model_Price'] = model_prices
fit['Pricing_Error'] = fit['Model_Price'] - fit['Market_Price']
fit['Absolute_Error'] = fit['Pricing_Error'].abs()
maturity_diagnostics = fit.groupby('Expiry').agg(
    Number_of_Quotes=('Pricing_Error', 'size'),
    Mean_Error=('Pricing_Error', 'mean'),
    MAE=('Absolute_Error', 'mean'),
    RMSE=('Pricing_Error', lambda values: float(np.sqrt(np.mean(values**2)))),
)
display(maturity_diagnostics)
display(fit[['Expiry','Strike','Market_Price','Model_Price','Pricing_Error']].head(10))


## 6. Calibration checkpoint

Do not treat the parameter estimates as final until we have checked sensitivity to initial values and integration settings, compared fitted NIG prices with Black–Scholes prices calibrated to the same market sample, and calculated the assignment's ARPE measure. Those robustness and benchmark steps follow after this first calibration pass.
